# ROGII Clean Artifact OOF Submission

This notebook produces a clean artifact-only OOF ensemble submission using only the competition input and `ravaghi/wellbore-geology-prediction-artifacts`.

It does not read target labels. It uses cached out-of-fold delta predictions from the artifact dataset, converts them to absolute TVT with `last_known_tvt`, averages the seven artifact models, and writes `/kaggle/working/submission.csv`.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

MODEL_NAMES = [
    "catboost-1",
    "catboost-2",
    "catboost-3",
    "lightgbm-1",
    "lightgbm-2",
    "lightgbm-3",
    "lightgbm-4",
]


def _candidate_paths():
    roots = []
    if Path("/kaggle/input").exists():
        roots.extend([
            Path("/kaggle/input/rogii-wellbore-geology-prediction"),
            Path("/kaggle/input/wellbore-geology-prediction-artifacts"),
            Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts"),
        ])
    roots.extend([
        Path("../data/raw/rogii-wellbore-geology-prediction"),
        Path("data/raw/rogii-wellbore-geology-prediction"),
        Path("../data/artifacts/wellbore-geology-prediction-artifacts"),
        Path("data/artifacts/wellbore-geology-prediction-artifacts"),
    ])
    return roots


def find_competition_root():
    for root in _candidate_paths():
        if (root / "sample_submission.csv").exists():
            return root
    if Path("/kaggle/input").exists():
        for sample_path in sorted(Path("/kaggle/input").glob("**/sample_submission.csv")):
            return sample_path.parent
    raise FileNotFoundError("Could not find competition sample_submission.csv")


def artifact_train_csv(root: Path) -> Path:
    nested = root / "data" / "train.csv"
    flat = root / "train.csv"
    if nested.exists():
        return nested
    if flat.exists():
        return flat
    return nested


def find_artifact_root():
    for root in _candidate_paths():
        if artifact_train_csv(root).exists():
            return root
    if Path("/kaggle/input").exists():
        for train_path in sorted(Path("/kaggle/input").glob("**/data/train.csv")):
            parent = train_path.parent.parent
            if any((parent / "models" / name / "oof_preds.pkl").exists() for name in MODEL_NAMES):
                return parent
    raise FileNotFoundError("Could not find artifact dataset with data/train.csv")


COMP_ROOT = find_competition_root()
ART_ROOT = find_artifact_root()
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

print(f"Competition root: {COMP_ROOT}")
print(f"Artifact root:    {ART_ROOT}")
print(f"Work root:        {WORK_ROOT}")

In [ ]:
def load_oof_delta(model_name: str) -> np.ndarray:
    nested = ART_ROOT / "models" / model_name / "oof_preds.pkl"
    flat = ART_ROOT / f"{model_name}_oof_preds.pkl"
    if nested.exists():
        path = nested
    elif flat.exists():
        path = flat
    else:
        raise FileNotFoundError(f"Missing OOF artifact for {model_name}")
    arr = joblib.load(path)
    return np.asarray(arr, dtype=np.float32)


sample = pd.read_csv(COMP_ROOT / "sample_submission.csv")
train_map = pd.read_csv(
    artifact_train_csv(ART_ROOT),
    usecols=["id", "last_known_tvt"],
    dtype={"id": "string", "last_known_tvt": "float32"},
)

if train_map["id"].duplicated().any():
    raise ValueError("Artifact train.csv has duplicate ids")

last_known = train_map["last_known_tvt"].to_numpy(np.float32)
pred_sum = np.zeros(len(train_map), dtype=np.float32)
loaded = []

for model_name in MODEL_NAMES:
    delta = load_oof_delta(model_name)
    if len(delta) != len(train_map):
        raise ValueError(f"{model_name} length mismatch: {len(delta)} != {len(train_map)}")
    pred_sum += last_known + delta
    loaded.append(model_name)
    print(f"Loaded {model_name}: {len(delta):,} rows")

artifact_pred = pd.DataFrame({
    "id": train_map["id"].astype(str),
    "tvt": pred_sum / np.float32(len(loaded)),
})

submission = sample[["id"]].merge(artifact_pred, on="id", how="left")
missing = int(submission["tvt"].isna().sum())
if missing:
    missing_ids = submission.loc[submission["tvt"].isna(), "id"].head(10).tolist()
    raise ValueError(f"Missing predictions for {missing} sample ids; examples: {missing_ids}")

if not np.isfinite(submission["tvt"].to_numpy(dtype=np.float64)).all():
    raise ValueError("Submission contains non-finite tvt values")

out_path = WORK_ROOT / "submission.csv"
submission.to_csv(out_path, index=False)

report = {
    "members": loaded,
    "artifact_rows": int(len(train_map)),
    "submission_rows": int(len(submission)),
    "ids_match_sample_order": bool(submission["id"].equals(sample["id"])),
    "missing_predictions": missing,
    "tvt_min": float(submission["tvt"].min()),
    "tvt_max": float(submission["tvt"].max()),
    "tvt_mean": float(submission["tvt"].mean()),
    "output": str(out_path),
}
(WORK_ROOT / "validation_report.json").write_text(json.dumps(report, indent=2))

print(json.dumps(report, indent=2))
print(submission.head())